# Day-1 evening smoke test — does the consistency formula even work?

Owner: Person D.

Goal: Plug **ground-truth queries** into the scorer on the 600-pair eval set.
If AUC with GT queries ≥ 0.75, the formula is sound and M2 quality will only modulate it.
If AUC with GT queries ≈ 0.5, the formula is broken — investigate before training M2.

This is risk **R1** from FINAL_DESIGN section 9. Must pass before Day 2.

In [ ]:
import sys, json
from pathlib import Path
PROJECT = Path('.').resolve()
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import numpy as np
import torch
from tqdm.auto import tqdm

from src.utils.io import load_jsonl, save_jsonl, load_yaml
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig
from src.eval.metrics import summarize_separation, confusion_at_threshold

cfg = load_yaml('configs/consistency.yaml')
ScoringConfig.__init__.__defaults__  # type check
scoring_cfg = ScoringConfig(alpha=cfg['alpha'], tau=cfg['tau'],
                             top_p=cfg['top_p'],
                             chunk_window=cfg['chunk_window'],
                             chunk_stride=cfg['chunk_stride'],
                             damp_with_fg=cfg['damp_with_fg'])

m1 = MomentDETRFeaturePredictor(
    ckpt_path=cfg['paths']['m1_ckpt'],
    repo_root=cfg['paths']['repo_root'],
)
scorer = ConsistencyScorer(m1, cfg['paths']['feature_dir'], scoring_cfg)
print('Scorer ready.')

In [ ]:
pairs = load_jsonl('processed/consistency_eval.jsonl')
print(f'Eval pairs: {len(pairs)}')

# Use the THUMB's query (positives) or the THUMB's query against a DIFFERENT video
# (negatives). The thumb_query field is the GT for the thumbnail-side anchor.
results = []
for r in tqdm(pairs):
    out = scorer.score(
        thumbnail_path=r['frame_path'],
        vid=r['video_vid'],
        queries=[r['thumb_query']],
    )
    results.append({
        'pair_id': r['pair_id'],
        'label':   r['label'],
        'score':   out['C'],
        'queries': out['queries'],
    })

Path('outputs/predictions').mkdir(parents=True, exist_ok=True)
save_jsonl(results, 'outputs/predictions/consistency_gt_query.jsonl')

In [ ]:
scores = [r['score'] for r in results]
labels = [r['label'] for r in results]
summary = summarize_separation(scores, labels)
print(json.dumps(summary, indent=2))

median = float(np.median(scores))
print('\nConfusion at threshold = median:')
print(json.dumps(confusion_at_threshold(scores, labels, median), indent=2))

import matplotlib.pyplot as plt
import numpy as np
pos = [s for s, l in zip(scores, labels) if l == 1]
neg = [s for s, l in zip(scores, labels) if l == 0]
plt.hist(pos, bins=30, alpha=0.5, label=f'positive (n={len(pos)})')
plt.hist(neg, bins=30, alpha=0.5, label=f'negative (n={len(neg)})')
plt.legend(); plt.xlabel('C(T, V)'); plt.title(f"GT-query smoke test  AUC={summary['auc']:.3f}")
plt.savefig('outputs/figures/score_distribution_gt.png', dpi=150, bbox_inches='tight')
plt.show()